# TFM - Data Cleaning

## Objective

The purpose of this phase is to clean and prepare the raw dataset for subsequent analysis and modeling tasks.

This process includes handling missing values, removing duplicate records, correcting data types, standardizing variables, and validating data consistency. The cleaned dataset will be stored in the `data/processed_data/` directory while preserving the original raw data.

Notebook: 02_Data_Cleaning

Author: Ronald Báez

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import unicodedata

In [2]:
# Load raw datasets
batting = pd.read_csv("../01_data/01_raw_data/cleaned_batting_stats.csv")

salary = pd.read_csv("../01_data/01_raw_data/mlb_salary_data.csv", encoding="latin1")

# Create original copy before cleaning
batting_original = batting.copy()
salary_original = salary.copy()

In [3]:
display(batting.shape)
display(salary.shape)

(4502, 36)

(13954, 4)

In [4]:
batting.columns

Index(['Rk', 'Player', 'Age', 'Team', 'Lg', 'WAR', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'BA', 'OBP', 'SLG',
       'OPS', 'OPS+', 'rOBA', 'Rbat+', 'TB', 'GIDP', 'HBP', 'SH', 'SF', 'IBB',
       'Pos', 'Awards', 'Year', 'RowNum'],
      dtype='object')

### Normalize Column Names

Column names are standardized to improve readability, consistency, and ease of use during the following phases of the project.

In [5]:
# Store original column names for reference
batting_original_columns = batting.columns.tolist()
salary_original_columns = salary.columns.tolist()

# Function to standardize column names
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace("+", "_plus", regex=False)
        .str.replace(r"[#*]", "", regex=True)
        .str.replace(r"[^a-zA-Z0-9_]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    return df

# Apply column name cleaning
batting = clean_column_names(batting)
salary = clean_column_names(salary)

# Rename salary player column for consistency with batting dataset
salary = salary.rename(columns={"name": "player"})

# Display updated column names
print("Batting columns:")
print(batting.columns)

print("\nSalary columns:")
print(salary.columns)

Batting columns:
Index(['rk', 'player', 'age', 'team', 'lg', 'war', 'g', 'pa', 'ab', 'r', 'h',
       '2b', '3b', 'hr', 'rbi', 'sb', 'cs', 'bb', 'so', 'ba', 'obp', 'slg',
       'ops', 'ops_plus', 'roba', 'rbat_plus', 'tb', 'gidp', 'hbp', 'sh', 'sf',
       'ibb', 'pos', 'awards', 'year', 'rownum'],
      dtype='object')

Salary columns:
Index(['year', 'team', 'player', 'salary'], dtype='object')


### Remove Special Characters and Fix Encoding Issues

Text values are reviewed to remove unnecessary symbols and correct common encoding issues that may appear in player names or categorical variables.

In [6]:
# Function to fix common encoding problems and remove accents
def normalize_text_encoding(value):
    if isinstance(value, str):
        try:
            value = value.encode("latin1").decode("utf-8")
        except:
            pass

        #remove accents and keep standard alphabetic characters
        value = unicodedata.normalize("NFKD", value)
        value = value.encode("ascii", "ignore").decode("utf-8")

    return value

In [7]:
# Identify text columns in both datasets
batting_text_columns = batting.select_dtypes(include="object").columns
salary_text_columns = salary.select_dtypes(include="object").columns

# Apply encoding normalization to batting dataset
for column in batting_text_columns:
    batting[column] = batting[column].apply(normalize_text_encoding)

# Apply encoding normalization to salary dataset
for column in salary_text_columns:
    salary[column] = salary[column].apply(normalize_text_encoding)

In [8]:
# Check if suspicious encoding characters remain in player names
batting[batting["player"].str.contains("Ã|Â|�", na=False)]["player"].unique()


array([], dtype=object)

In [9]:
salary[salary["player"].str.contains("Ã|Â|�", na=False)]["player"].unique()

array([], dtype=object)

In [10]:
# Check suspicious encoding characters in all text columns
suspicious_pattern = "Ã|Â|�"

for dataset_name, dataset in [("batting", batting), ("salary", salary)]:
    print(f"\nDataset: {dataset_name}")
    
    text_columns = dataset.select_dtypes(include="object").columns
    
    for column in text_columns:
        suspicious_values = dataset[
            dataset[column].str.contains(suspicious_pattern, na=False)
        ][column].unique()
        
        if len(suspicious_values) > 0:
            print(f"\nColumn: {column}")
            print(suspicious_values)


Dataset: batting

Dataset: salary


### Remove Unnecessary Characters and Spaces

Unnecessary symbols and extra spaces are removed from text columns to improve consistency and avoid matching issues in later stages.

In [11]:
batting_text_columns = batting.select_dtypes(include="object").columns
salary_text_columns = salary.select_dtypes(include="object").columns

In [12]:
# Remove unnecessary symbols and extra spaces in batting dataset
for column in batting_text_columns:
    batting[column] = (
        batting[column]
        .str.replace("*", "", regex=False)
        .str.replace("#", "", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Remove unnecessary symbols and extra spaces in salary dataset
for column in salary_text_columns:
    salary[column] = (
        salary[column]
        .str.replace("*", "", regex=False)
        .str.replace("#", "", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [13]:
# Check if unnecessary symbols remain in player names
batting[batting["player"].str.contains(r"\*|#", na=False)]["player"].unique()

array([], dtype=object)

###  Standardize Key Text Values

Key text variables are standardized to ensure consistent formatting across datasets, especially player names, team identifiers, and league identifiers.

In [14]:
# Standardize player name
if "player" in batting.columns:
    batting["player"] = batting["player"].str.lower()

if "player" in salary.columns:
    salary["player"] = salary["player"].str.lower()

# Standardize team and league identifiers in batting dataset
for item in ["team", "lg"]:
    if item in batting.columns:
        batting[item] = batting[item].str.upper()

# Standardize team identifier in salary dataset
salary["team"] = salary["team"].str.upper()

In [15]:
# Check standardized key text values
print(batting[["player", "team", "lg"]].head())

print("\n", salary[["player", "team"]].head())

            player team   lg
0       a.j. ellis  2TM   NL
1       aj pollock  2TM  2LG
2      aaron hicks  2TM   AL
3       aaron hill  2TM  2LG
4  abraham almonte  2TM  2LG

           player team
0  kelly johnson  ARI
1   joe saunders  ARI
2    chris young  ARI
3   stephen drew  ARI
4   justin upton  ARI


In [16]:
batting.columns

Index(['rk', 'player', 'age', 'team', 'lg', 'war', 'g', 'pa', 'ab', 'r', 'h',
       '2b', '3b', 'hr', 'rbi', 'sb', 'cs', 'bb', 'so', 'ba', 'obp', 'slg',
       'ops', 'ops_plus', 'roba', 'rbat_plus', 'tb', 'gidp', 'hbp', 'sh', 'sf',
       'ibb', 'pos', 'awards', 'year', 'rownum'],
      dtype='object')

###  Convert Data Types

Columns are converted to their appropriate data types to ensure that years, salaries, and baseball statistics can be processed correctly in later stages.

In [17]:
# Convert year columns to numeric format
batting["year"] = pd.to_numeric(batting["year"], errors="coerce")
salary["year"] = pd.to_numeric(salary["year"], errors="coerce")

# Convert salary column to numeric format
salary["salary"] = (
    salary["salary"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

salary["salary"] = pd.to_numeric(salary["salary"], errors="coerce")




In [18]:
# Define non-numeric columns in batting dataset
batting_text_columns = ["player", "team", "lg", "pos", "awards"]

# Select all columns that should be numeric
batting_numeric_columns = batting.columns.difference(batting_text_columns)

# Convert batting statistics to numeric format
batting[batting_numeric_columns] = batting[batting_numeric_columns].apply(
    pd.to_numeric, errors="coerce"
)

###  Handle Missing Values

Missing values are reviewed to identify incomplete records and decide whether they should be removed, kept, or documented depending on their relevance for the project.

In [19]:
# Check missing values in salary dataset
salary_missing = salary.isnull().sum()
salary_missing = salary_missing[salary_missing > 0]

salary_missing

Series([], dtype: int64)

In [26]:
# Check missing values in batting dataset

batting_missing_before = batting_original.isnull().sum()
batting_missing_before = batting_missing_before[batting_missing_before > 0]

batting_missing_before

Awards    3758
dtype: int64

### Missing Values Treatment

The `awards` column contains missing values because most players did not receive individual awards. These missing values are treated as meaningful absence of awards and replaced with `no_award`.

In [27]:
# Replace missing awards with a meaningful category
batting["awards"] = batting["awards"].fillna("no_award")

In [24]:
# Check missing values after treatment
batting_missing_after = batting.isnull().sum()
batting_missing_after = batting_missing_after[batting_missing_after > 0]

batting_missing_after

Series([], dtype: int64)

###  Review Duplicate Records

Duplicate records are reviewed to identify repeated rows and potential player-season cases. In baseball data, a player may appear more than once in the same season if he played for multiple teams.

In [32]:
# Check exact duplicate rows in both datasets
batting_duplicates = batting.duplicated().sum()
salary_duplicates = salary.duplicated().sum()

print(f"Exact duplicate rows in batting dataset: {batting_duplicates}")
print(f"Exact duplicate rows in salary dataset: {salary_duplicates}")

Exact duplicate rows in batting dataset: 0
Exact duplicate rows in salary dataset: 1


In [ ]:
# Remove exact duplicate rows from salary dataset
salary = salary.drop_duplicates()

In [34]:
# Check exact duplicate rows after removal
salary.duplicated().sum()

np.int64(0)

###  Validate Salary Values

Salary values are validated to ensure they are numeric, non-missing, and greater than zero.

In [35]:
# Check basic salary statistics
salary["salary"].describe()

count    1.395300e+04
mean     3.301938e+06
std      5.396146e+06
min      2.923000e+03
25%      4.800000e+05
50%      7.400000e+05
75%      3.725000e+06
max      5.500000e+07
Name: salary, dtype: float64

In [36]:
# Check missing, zero, or negative salary values
invalid_salary_values = salary[
    (salary["salary"].isnull()) | 
    (salary["salary"] <= 0)
]

invalid_salary_values

,year,team,player,salary


In [37]:
# Count invalid salary values
invalid_salary_values.shape[0]

0

###  Final Consistency Check

A final consistency check is performed to confirm that both cleaned datasets have no missing values, no duplicate records, and valid dimensions before export.

In [38]:
# Final consistency check
print(f"Batting shape: {batting.shape}")
print(f"Salary shape: {salary.shape}")

print(f"\nMissing values in batting: {batting.isnull().sum().sum()}")
print(f"Missing values in salary: {salary.isnull().sum().sum()}")

print(f"\nDuplicate rows in batting: {batting.duplicated().sum()}")
print(f"Duplicate rows in salary: {salary.duplicated().sum()}")

Batting shape: (4502, 36)
Salary shape: (13953, 4)

Missing values in batting: 0
Missing values in salary: 0

Duplicate rows in batting: 0
Duplicate rows in salary: 0


##  Save Cleaned Datasets

The cleaned datasets are saved in the `data/processed_data/` folder for use in the next phases of the project.

In [45]:
# Save cleaned datasets
batting.to_csv("../01_data/02_processed_data/batting_clean.csv", index=False)
salary.to_csv("../01_data/02_processed_data/salary_clean.csv", index=False)
print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


##  Conclusions

In this phase, both raw datasets were cleaned and prepared for the next stages of the project.

The main cleaning tasks included standardizing column names, fixing encoding issues, removing unnecessary characters and spaces, converting data types, handling missing values, removing duplicate records, and validating salary values.

As a result, two cleaned datasets were generated and saved in the `01_data/02_processed_data/` folder:

- `batting_clean.csv`
- `salary_clean.csv`

These datasets are now ready for the next phase of the project.